# 以 ollama 串接 langchain 執行 llama3-8b

In [6]:
# 設定必要的資源包和函式
# %pip install langchain chromadb pypdf fastembed
# %pip install aiofiles
# %pip install langchain_google_genai
# %pip install langchain_community

In [7]:
import os
from IPython.display import display
from IPython.display import Markdown
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy
import json

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain.schema.output_parser import StrOutputParser
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import PromptTemplate
from langchain.vectorstores.utils import filter_complex_metadata
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

# 設定模型參數

In [8]:
model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0
chunk_size=1024
chunk_overlap=10
# os.environ["GOOGLE_API_KEY"] = os.getenv('GEMINI_API_KEY')

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

# RAG

In [9]:
# 用來讀取文字檔，並轉成 embedding
from typing import AsyncIterator, Iterator

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document


class CustomDocumentLoader(BaseLoader):
    """An example document loader that reads a file line by line."""

    def __init__(self, file_path: str) -> None:
        """Initialize the loader with a file path.

        Args:
            file_path: The path to the file to load.
        """
        self.file_path = file_path

    def lazy_load(self) -> Iterator[Document]:  # <-- Does not take any arguments
        """A lazy loader that reads a file line by line.

        When you're implementing lazy load methods, you should use a generator
        to yield documents one by one.
        """
        with open(self.file_path, encoding="utf-8") as f:
            line_number = 0
            for line in f:
                yield Document(
                    page_content=line,
                    metadata={"line_number": line_number, "source": self.file_path},
                )
                line_number += 1

    # alazy_load is OPTIONAL.
    # If you leave out the implementation, a default implementation which delegates to lazy_load will be used!
    async def alazy_load(
        self,
    ) -> AsyncIterator[Document]:  # <-- Does not take any arguments
        """An async lazy loader that reads a file line by line."""
        # Requires aiofiles
        # Install with `pip install aiofiles`
        # https://github.com/Tinche/aiofiles
        import aiofiles

        async with aiofiles.open(self.file_path, encoding="utf-8") as f:
            line_number = 0
            async for line in f:
                yield Document(
                    page_content=line,
                    metadata={"line_number": line_number, "source": self.file_path},
                )
                line_number += 1


In [10]:
# RAG
class RAG:
    vector_store = None
    retriever = None
    chain = None

    def __init__(self):
        self.model = llm
        self.text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.prompt = PromptTemplate.from_template(
            """
            <s> [INST] 以下是今天的新聞資訊，請對於明天開盤時買進收盤時賣出的獲利可能性進行判斷，如果可以獲利請回答#好 否則#不好 如果與股市無關則回答#無關 [/INST] </s> 
            [INST] 問題: {question} 
            文章: {context} 
            回答: [/INST]
            """
        )

    def ingest(self, pdf_file_path: str):
        docs = CustomDocumentLoader(file_path=pdf_file_path).load()
        chunks = self.text_splitter.split_documents(docs)
        chunks = filter_complex_metadata(chunks)
        # chunks.append(chunks)

        vector_store = Chroma.from_documents(documents=chunks, embedding=FastEmbedEmbeddings())
        self.retriever = vector_store.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs={
                "k": 3,
                "score_threshold": 0.5,
            },
        )

        self.chain = ({"context": self.retriever, "question": RunnablePassthrough()}
                      | self.prompt
                      | self.model
                      | StrOutputParser())

    def ask(self, query: str):
        if not self.chain:
            return "Please, add a document first."

        return self.chain.invoke(query)

    def clear(self):
        self.vector_store = None
        self.retriever = None
        self.chain = None


In [11]:
# 測試rag是否能夠正常運行
# import os

# rag = RAG()
# rag.clear()
# rag.ingest(os.path.dirname(os.path.abspath(os.getcwd()))  + "/stock_news/3023/3023_news/20050129")

# # response = rag.ask(f"請對{stock_id} {stock_names[idx]}的新聞對的隔日開盤買進收盤賣出進行判斷，如果可以獲利請回答#好 否則#不好 如果新聞內容與股市無關則回答#無關")
# response = rag.ask(f"請找出對3023信邦的股價有影響的新聞資料")

# print(response)

In [12]:
# rag 回應
def rag_response(ragfile, instruction):
    '''
        ragfile: string 檔案路徑
        instruction: string 問題指令
        
        return: string 回答
    '''
    text = ''
    try:
        rag = RAG()
        rag.clear()
        rag.ingest(ragfile)
        text = rag.ask(instruction)
        # print(text)
    except Exception as e:
        print(e)
        pass
    return text

In [13]:
def llm_response(filepath, question):
    ''' 
        filepath: string 檔案路徑
        question: string 問題指令
        
        return: string 回答
    '''
    with open(filepath, 'r', encoding='utf-8') as f:
        news = f.read()

    prompt = f"""<s> [INST] 以下是今天的新聞資訊，請對於明天開盤時買進收盤時賣出的獲利可能性進行判斷，如果可以獲利請回答#好 否則#不好 如果與股市無關則回答#無關 [/INST] </s> 
            [INST] 問題: {question} 
            文章: {news} 
            回答: [/INST]
            """

    try:
        response = llm.invoke(prompt)
        # print(text)
        return response.content
    except Exception as e:
        print(e)
        pass
    
    return "無關"
    

In [14]:
# 轉換回答變成訊號
def response_to_signal(text):
    '''
        text: string 回答訊息
        
        return: int 回答訊號
    '''
    if text.find('不好') > 0:
        signal = -1
    elif text.find('好') > 0:
        signal = 1
    else:
        signal = 0

    return signal

In [15]:
# 計算隔天當沖準確率（開盤買進收盤賣出or開盤賣出收盤回補）
def eval(signal_l, price):
    '''
        signal: list, [日期, 評分]
        price: dataframe, 股票價格

        return: float, float 準確率, 平均報酬率
    '''
    tp = 0
    tn = 0
    fp = 0
    fn = 0
    rtns = []   # 記錄當日沖銷的報酬率

    for idx, item in enumerate(signal_l):
        signal = float(item[1])
        index_location = price.index.get_loc(item[0])  # 獲取特定索引的位置
        if index_location == len(price.index) - 1: # 避免超出索引
            break
        next_index = price.index[index_location + 1]  # 獲取下一個索引的位置
        next_row = price.loc[next_index]  # 找到下一個索引的列
        open = float(next_row['Open'])
        close = float(next_row['Close'])
        
        if signal > 0:
            rtns.append(close/open)
            if close > open:
                tp += 1
            elif close < open:
                fp += 1
            else:   
                pass

        elif signal < 0:
            rtns.append(open/close)
            if close < open:
                tn += 1
            elif close > open:
                fn += 1
            else:   
                pass
    if (tp + tn + fp + fn)  == 0:
        accuracy = 0
    else:
        accuracy = (tp + tn) / (tp + tn + fp + fn) # 準確率
        print(tp, tn, fp, fn)

    if len(rtns) == 0:
        avg_rtn = 1.0
    else:
        avg_rtn = sum(rtns)/len(rtns)  # 這檔股票在這段時間的平均報酬率
        print(avg_rtn)
    return accuracy, avg_rtn

In [16]:
def write_output(content, folder_path):
    '''
        json_content: string, json格式的資料
        raw_data: list, 評分的資料
        folder_path: string, 資料夾路徑
    '''
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    with open(folder_path + "detail.json" , "w", encoding="UTF-8") as f:
        f.write(content)
    print("寫入" + folder_path)

# RAG主程式

In [19]:
# 上市公司股票代號
# 代號 ['2618', '2881', '2880', '2211', '2731', '2707', '3023', '9941', '2308', '1514', '1101', '2542', '3036', '2385', '1303', '2912', '2323', '2603', '1419', '1604', '2486', '5269', '1314', '8033', '3017', '2903', '9943', '2368', '9904', '1102', '3231', '3004', '3035', '6176', '1103', '2609', '2002']
# 對應名稱 ['長榮航', '富邦金', '華南金', '長榮鋼', '雄獅', '晶華', '信邦', '裕融', '台達電', '亞力', '台泥', '興富發', '文曄', '群光', '南亞', '統一超', '中環', '長榮', '新光金', '聲寶', '一詮', '祥碩', '中石化', '遠雄', '奇鋐', '遠百', '好樂迪', '金像電', '寶成', '亞泥', '緯創', '豐達科', '智原', '瑞儀', '嘉泥', '陽明', '中鋼']
start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

stock_ids = ['2618', '2881', '2880', '2211', '2731', '2707', '3023', '9941', '2308', '1514']
stock_names = ['長榮航', '富邦金', '華南金', '長榮鋼', '雄獅', '晶華', '信邦', '裕融', '台達電', '亞力']

In [ ]:
# count = 0
for idx, stock_id in enumerate(stock_ids):

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    acc_ = 0
    print(f'正在執行{stock_id} {stock_names[idx]}')
    qurey = f"請找出對{stock_id}{stock_names[idx]}的股價有影響的新聞資料"
    signal = []
    counter = 0
    for date in df_price.index:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(date, '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        
        sig = 0
        for index, file_name in enumerate(sorted_news_files):
            if file_name == date:
                res = rag_response(folder_path + file_name, qurey)   # file_name, qurey
                sig = response_to_signal(res)    # file_name, qurey
                # count += 1
                break
        signal.append([date, sig]) # [日期, 訊號]
        print(f'{date}: {sig}')

    acc, rtn = eval(signal, df_price) # 得到這檔股票的準確率和報酬率

    # 輸出結果 json file
    output_data = {
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "acc": acc,
        "rtn": rtn,
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "raw_data": signal
    }

    # 轉換成 JSON 格式
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
    write_output(content=json_content, folder_path="eval_rag/output052801rag/" + stock_id + "/")
# print(count)

# 沒有RAG 當作對照組

In [20]:
llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY_2'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })
for idx, stock_id in enumerate(stock_ids):

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    acc_ = 0
    print(f'正在執行{stock_id} {stock_names[idx]}')
    prompt = f"請找出對{stock_id}{stock_names[idx]}的股價有影響的新聞資料"
    signal = []
    counter = 0
    for date in df_price.index:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(date, '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        
        sig = 0
        for index, file_name in enumerate(sorted_news_files):
            if file_name == date:
                res = llm_response(folder_path + file_name, prompt)
                print(res)
                sig = response_to_signal(res) 
                break
        
        signal.append([date, sig]) # [日期, 訊號]
        print(f'{date}: {sig}')

    acc, rtn = eval(signal, df_price) # 得到這檔股票的準確率和報酬率

    # 輸出結果 json file
    output_data = {
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "acc": acc,
        "rtn": rtn,
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "raw_data": signal
    }

    # 轉換成 JSON 格式
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
    write_output(content=json_content, folder_path="eval_rag/output_norag052801/" + stock_id + "/")

正在執行2881 富邦金
20230103: 0
20230104: 0
20230105: 0
#好 

20230106: 1
回答: #好 

**理由:**

新聞中提到「富邦金」等個股開盤上漲，且指數大漲逾200點，收復14600點。這表明市場對金融股普遍看好，富邦金作為金融龍頭，其股價有望受益於市場的積極氛圍。 

**注意事項:**

* 新聞僅提供短期市場趨勢的參考，並非投資建議。
* 投資決策應基於更全面的分析，包括公司基本面、市場狀況等。
* 投資有風險，需謹慎評估。 

20230109: 1
20230110: 0
#不好 

**理由:**

1. **富邦金2022年12月虧損154.9億，為5家虧損金控之首，且全年獲利大幅減少近千億元。** 這顯示富邦金目前財務狀況不佳，對股價不利。
2. **富邦金旗下富邦人壽和富邦產險，12月分別虧損73.89億和91.05億，主要原因是防疫險理賠。** 防疫險理賠的影響尚未完全消除，未來可能仍會對富邦金造成負面影響。
3. **雖然富邦金2022年全年獲利467.18億，仍為獲利，但與2021年相比大幅縮水。** 這顯示富邦金的盈利能力正在下降。

綜合以上資訊，富邦金目前面臨多項不利因素，預計短期內股價可能走低，因此不建議在明天開盤時買入。 

20230111: -1
#不好 

富邦金的股價受到多個因素影響，新聞中提到了以下幾個可能影響富邦金股價的事件：

1. **金金併:** 新聞提到金管會最新試算，目前具有發動敵意金金併的資格已從2020年的「5金6銀」，增加為2022年的「6金7銀」，而富邦金是其中之一。這意味著富邦金可能會成為其他金控的併購目標，可能會對股價造成一定程度的波動。
2. **員工跳槽:** 新聞提到yes123求職網的調查，顯示高達862萬勞工想在年後換工作，比率創14年新高，而富邦金是員工最想去的企業之一。這表示富邦金可能會面臨人才流失的風險，可能會對其業務造成一定程度的影響，進而影響股價。

然而，新聞中並沒有明確指出富邦金的股價會因此上漲或下跌。因此，無法判斷明天開盤時買進收盤時賣出是否能獲利。 

20230112: -1
20230113: 0
#好 

富邦金（2881）在新聞中被提及，且有以下正面訊息：

* **外資買超：**富邦金位列外資買超前十大個

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


#好 

**理由:**

* **第一季獲利亮眼：**富邦金第一季稅後盈餘續居金控業之冠，每股稅後純益（EPS）1.13元，顯示公司營運狀況良好。
* **強化資本結構：**富邦金控和富邦人壽計畫發行公司債，充實資本，強化資本結構，有助於提升投資人信心。
* **分紅保單布局：**富邦人壽推出分紅保單，有助於提升公司長期獲利能力。
* **防疫險風暴接近尾聲：**富邦產險防疫險理賠金額接近尾聲，對公司營運的影響逐漸減弱。

**總體而言，富邦金近期利好消息較多，預計股價會受到正面影響。** 

20230518: 1


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


#不好 

新聞中提到了富邦金控和富邦人壽計畫發行公司債，以及富邦產險第一季的虧損，這些消息都可能對富邦金的股價造成負面影響。因此，明天開盤時買進收盤時賣出，獲利可能性較低。 

20230519: -1
20230522: 0
#不好 

**理由：**

* 文章中提到富邦金在台股權值股拉回，以及電金拉回，指數跌逾40點的訊息。 
* 這些訊息都表明富邦金的股價在當天有下跌的趨勢，因此明天開盤買進收盤賣出可能無法獲利。 

20230523: -1
回答: #不好 

**理由:**

新聞中提到富邦金開盤下跌，並且金融族群整體走弱，這表示市場對富邦金的短期展望並不樂觀。 因此，在這種情況下，明天開盤買進收盤賣出獲利的可能性不高。 

20230524: -1
20230525: 0
#無關 

這篇文章主要講的是國泰金控的第一季法說會，內容包含國泰金的投資策略、財務狀況以及對未來經濟的展望。雖然文章有提到富邦金的股價，但並未提供任何關於富邦金的資訊，也沒有分析富邦金的財務狀況或未來前景。因此，這篇文章無法提供判斷富邦金股價走勢的依據。

20230526: 0
#有關係

這篇文章提到了富邦金（2881）的獨董薪酬級距曾高達1,000~1,500萬元間。 

這條資訊可能影響股價，因為：

* **獨董薪酬高可能反映公司治理狀況良好，吸引高階人才加入，對公司發展有利。**  這可能導致投資者對富邦金的信心增加，進而推升股價。
* **獨董薪酬高也可能引發市場對公司治理的質疑，認為公司過度重視獨董酬勞，而忽略了其他方面的發展。**  這可能導致投資者對富邦金的信心降低，進而壓低股價。

因此，這篇文章的資訊對富邦金（2881）的股價有潛在的影響，但最終的影響方向需要根據市場反應和更多資訊來判斷。

20230529: 1
回答: #有關 

新聞提到富邦金開盤上漲，這表示富邦金的股價有上升的趨勢，可能對股價有正面影響。 

20230530: 0
20230531: 0
20230601: 0
#好 

20230602: 1
回答: #有關係 

新聞提到富邦金開盤上漲，代表市場對富邦金的股價持正面看法。 

20230605: 0
20230606: 0
20230607: 0
20230608: 0
#不好 

**理由:**

雖然新聞中提到富邦金去年獲

...

#好 
信邦（3023）加入歐盟可更換電池機車聯盟（SBMC），協助制訂連接器規格，看好明年新能源車商機。 
這條新聞顯示信邦在電動車領域的發展潛力，可能有利於股價。 

20231218: 1
20231219: 0
#無關 

這則新聞資訊是關於太康精密子公司背書保證超過限額的公告，而3023信邦為光纖網路設備製造商，與太康精密並無直接關聯，因此這則新聞對信邦股價影響不大。 

20231220: 0
20231221: 0
20231222: 0
20231225: 0
20231226: 0
20231227: 0
20231228: 0
回答: #不好 

**理由:**

信邦電子董事葉辛池辭職，屬於公司內部人事變動，通常對股價影響較小，尤其是辭職原因是個人因素，沒有涉及公司經營層面的重大變化。因此，僅憑此消息很難預測明天開盤時買進收盤時賣出是否能獲利。

20231229: -1
23 7 9 6
1.0005793612506377
寫入eval_rag/output_norag052801/3023/
正在執行9941 裕融
20230103: 0
20230104: 0
#無關 

這則新聞是關於裕融公司一○九年度第一期有擔保普通公司債的到期和下櫃公告，與裕融股票的價格沒有直接關係。 

20230105: 0
20230106: 0
#好 

**原因:**

裕融公布 2022 年 12 月自結合併損益，稅後盈餘 334.7 億元，EPS 為 0.61 元，年增 12.6%。 累積 1-12 月稅後盈餘 6,124.7 億元，EPS 為 12.05 元，年增 0.97 元。  

**說明:**

* 裕融的營收和獲利都呈現成長，顯示公司營運狀況良好，這對股價通常會有正面影響。 
* 投資者可能會對裕融的財務表現感到樂觀，進而推升股價。

**注意事項:**

* 投資決策應根據多方面的分析，不應只根據單一新聞資訊。
* 股市存在風險，投資前應做好功課，並做好風險管理。

20230109: 1
20230110: 0
#無關 

新聞內容提到裕融租賃總經理職務調整以及董事會改選，但這些變動不直接影響裕融的股價。 

20230111: 0
20230112: 0
20230113: 0
20230116: 0
#無關 

這則新聞是裕融企業

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


#不好 

外資賣超亞力(1514-TW) 逾 2800 張，顯示外資對該股票不看好，短期內股價可能會下跌。 

20230726: -1
#好

**理由：**

亞力電機發行無擔保轉換公司債，目的是用於償還金融機構借款。這表示公司正在積極改善財務狀況，這可能對股價產生正面影響。

**其他因素：**

* 債券發行金額為新台幣10億元，顯示公司對未來發展充滿信心。
* 轉換公司債可以轉換為普通股，這可能會吸引投資者，進而推升股價。

**注意：**

* 股市波動很大，以上僅供參考，不構成投資建議。
* 建議您在投資前，應自行做更深入的研究，並考慮自身的風險承受能力。

20230727: 1
20230728: 0
20230731: 0
20230801: 0
20230802: 0
20230804: 0
20230807: 0
#好 

亞力電機公告第二季財報，受惠台電強韌電網計畫需求強勁，獲利再創新高，且下半年半導體廠設備陸續出貨，第三、四季營收獲利可望再飆新高。此外，亞力取得竹科升壓站的20 億元大單，在手訂單衝破 100 億元，今年營收、獲利可望年增 2 成，優於原先預期。這些消息都預示著亞力電機的未來發展潛力，預計將帶動股價上漲。 

20230808: 1
回答: 
- 亞力(1514-TW)股價下跌4.3元，跌幅7.11%，與大盤表現同步。
- 該公司所屬產業為電機機械，主要業務為配電盤、電機器材、電子產品、變壓器。 
- 缺乏關於亞力(1514-TW)的最新相關新聞，無法判斷對股價的影響。 

20230809: 0
回答: #好 

20230810: 1
#好 

**理由:**

新聞提到台電釋出千億元以上的強韌電網計畫商機，亞力（1514）受益，在手訂單突破百億元，第二季及上半年營收獲利均飆同期高，上半年EPS達到1.28元。 

**說明:**

* 台電釋出的千億元強韌電網計畫商機，預計將帶動相關電機設備廠的營收成長。
* 亞力作為電機設備廠，在手訂單突破百億元，且營收獲利飆升，顯示其已成功獲得台電計畫的訂單，未來將持續受益於該計畫。 
* 上半年EPS達到1.28元，顯示其盈利能力強勁。

**需要注意:**

* 股市瞬息萬變，以上僅為分析，不代表實際投資建議。
* 投資前請務必做好功課，並根據自身風險承受能力做出判斷。 


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


回答: 台電釋出汰舊換新設備及強韌電網計畫訂單，亞力（1514）將受益。 

20230911: 0
20230912: 0
20230913: 0
20230914: 0
20230915: 0
20230918: 0
回答: #無關 

**說明:** 

提供的文章內容主要包含亞力(1514-TW)的股價表現、成交量、產業資訊和近5日漲跌速等數據。這些資訊都是關於過去的股價走勢和市場數據，並未提供關於未來股價的預測或影響因素。 

因此，這篇文章無法提供判斷明日開盤時買進收盤時賣出獲利可能性所需的資訊。 

20230919: 0
#好 

**理由：**

文章提到亞力（1514）受惠台電強韌電網計畫題材，19日儘管三大法人合計賣超159張，但市場承接力道強勁，終場大漲3.9元、收53.8元，突破5日、10及20日均線反壓。

這顯示市場對亞力未來發展抱持樂觀態度，且短期內有向上突破的趨勢。雖然文章也提到上檔將面臨季線58.38元以上賣壓，但短期內突破均線的強勁動能，有可能帶動股價進一步上漲。

**風險：**

* 台電強韌電網計畫的實際進展可能不如預期。
* 市場對AI產業的疑慮可能影響整體科技股表現，進而波及亞力股價。
* 上檔季線壓力可能導致股價漲幅受限。

**結論：**

雖然存在一定的風險，但根據文章的資訊，亞力在短期內有較高的獲利可能性。 

**備註：**

投資決策需要綜合考慮多種因素，僅憑一篇新聞資訊不足以作為投資決策的唯一依據。 

20230920: 1
20230921: 0
回答: 
* 亞力(1514-TW)近5分K漲速2.4%，股價表現優於大盤。
* 近5日股價上漲3.7%，表現優於所屬產業電機機械類指數。
* 近5日籌碼最新相關新聞。 

20230922: 0
20230925: 0
20230926: 0
#有

文章提到「重電概念股成為撐盤要角」，並點名「(1514-TW) 亞力」在紅盤上，表示亞力股價有上漲的跡象。因此，對 1514 亞力的股價有影響的新聞資料存在。 

20230927: 0
20230928: 0
20231002: 0
#好 

**理由:**

新聞提到亞力（1514）業績升溫，在手訂單突破百億元，8月營收創歷史同期新高，且下半年營運有望優於上半年。法人預估今年全年營收將來到92億元，甚至上

In [ ]:
# id = '2211'
# with open("eval_llama/output052601 01-23-39-234/" + id + "/detail.json") as f:
#     data = json.load(f)
#     print(data['raw_data'])
# df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + id + "twse.csv", dtype=str)
# df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
# for d in data['raw_data']:
#     print(d)
#     print(eval(d, df_price))
#     print()